In [ ]:
# 1) Optional: installs (run once)
!pip install fredapi pandas numpy scikit-learn matplotlib seaborn xgboost tensorflow --quiet

In [ ]:
# 2) Imports + environment detection (GPU / cuML / XGBoost / TF)
import os, sys, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

# ML libs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import tensorflow as tf

# Env detection flags
USE_GPU = False
USE_CUML = False
USE_XGB = True
USE_TF = True
try:
    import torch
    if torch.cuda.is_available(): USE_GPU = True
except Exception:
    pass
try:
    import cuml
    USE_CUML = True; USE_GPU = True
except Exception:
    pass
print('USE_GPU=', USE_GPU, 'USE_CUML=', USE_CUML, 'USE_XGB=', USE_XGB, 'USE_TF=', USE_TF)

In [ ]:
# 3) Data fetch (FRED) — unchanged logic mostly
import pandas as pd
from fredapi import Fred

# LƯU Ý: Điền API key FRED của bạn vào đây
FRED_API_KEY = "ced7b9169f789e50e5658ba0fafce830"
fred = Fred(api_key=FRED_API_KEY)

series_map = {
    "NGDPRSAXDCUSQ": "United States",
    "NGDPRSAXDCDEQ": "Germany",
    "NGDPRSAXDCJPQ": "Japan",
    "NGDPRSAXDCCAQ": "Canada",
    "NGDPRSAXDCAUQ": "Australia",
    "NGDPRSAXDCITQ": "Italy",
    "NGDPRSAXDCFRQ": "France",
    "NGDPRSAXDCGBQ": "United Kingdom",
    "NGDPRNSAXDCINQ": "India",
    "NGDPRSAXDCMXQ": "Mexico",
    "NGDPRSAXDCKRQ": "South Korea",
    "NGDPRSAXDCBRQ": "Brazil"
}

all_rows = []
for sid, cname in series_map.items():
    try:
        s = fred.get_series(sid)  # pandas Series(datetime_index -> GDP)
        df_c = (
            s.to_frame(name="GDP")
             .reset_index()
             .rename(columns={"index": "Date"})
        )
        df_c["Country"] = cname
        all_rows.append(df_c)
        print(f"Fetched {cname}: {len(df_c)} rows")
    except Exception as e:
        print(f"Failed {cname}: {e}")

fred_df = pd.concat(all_rows, ignore_index=True)
fred_df["Date"] = pd.to_datetime(fred_df["Date"])
fred_df = fred_df.sort_values(["Country", "Date"]).reset_index(drop=True)

display(fred_df.head())
fred_df.to_csv("fred.csv", index=False)
print("✅ Saved fred.csv")
import os
import numpy as np
import pandas as pd

if not os.path.exists("fred.csv"):
    raise FileNotFoundError("fred.csv not found. Run Cell 1 first or provide fred.csv manually.")

fred_df = pd.read_csv("fred.csv")
fred_df["Date"] = pd.to_datetime(fred_df["Date"])
fred_df = fred_df.sort_values(["Country", "Date"]).reset_index(drop=True)

def preprocess_and_interpolate(df):
    df = df.sort_values(["Country", "Date"])
    df2 = (
        df.groupby("Country")
          .apply(lambda g: g.interpolate(method="linear").ffill().bfill())
          .reset_index(drop=True)
    )
    return df2

def create_features_and_targets(
    df,
    target_col="GDP",
    lag_list=[1,2,3,4,8,12,16],
    rolling_windows=[4,8,12,16],
    target_quarters=[4,16]
):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Country","Date"]).reset_index(drop=True)

    df["time_index"] = df.groupby("Country").cumcount()

    # Lags cho GDP
    for lag in lag_list:
        df[f"{target_col}_lag_{lag}"] = df.groupby("Country")[target_col].shift(lag)

    # Rolling stats cho GDP
    for w in rolling_windows:
        df[f"{target_col}_roll_mean_{w}"] = (
            df.groupby("Country")[target_col].transform(lambda x: x.rolling(w, min_periods=1).mean())
        )
        df[f"{target_col}_roll_std_{w}"] = (
            df.groupby("Country")[target_col].transform(lambda x: x.rolling(w, min_periods=1).std())
        )

    # Targets tương lai
    for h in target_quarters:
        df[f"target_q_{h}"] = df.groupby("Country")[target_col].shift(-h)
        df[f"target_q_{h}_growth_pct"] = (
            (df[f"target_q_{h}"] - df[target_col]) / df[target_col] * 100
        )

    # Giữ lại các hàng có đủ lag lớn nhất và target
    need_cols = [f"{target_col}_lag_{max(lag_list)}"] + [f"target_q_{h}" for h in target_quarters]
    df = df.dropna(subset=need_cols).reset_index(drop=True)
    return df

fred_clean = preprocess_and_interpolate(fred_df)
fred_processed_df = create_features_and_targets(
    fred_clean,
    target_col="GDP",
    lag_list=[1,2,3,4,8,12,16],
    rolling_windows=[4,8,12,16],
    target_quarters=[4,16]
)

print("fred_processed_df ready:")
display(fred_processed_df.head())
fred_processed_df.info()
from sklearn.preprocessing import StandardScaler
from typing import Dict, Any

def normalize_features_per_country(df, feature_cols, target_col):
    """
    Chuẩn hóa X và y theo từng Country.
    Trả về:
        df_norm: DataFrame có các cột feature_cols_norm + target_norm
        scalers: dict[country] = {"x": scaler_x, "y": scaler_y}
    """
    df_norm = df.copy()
    scalers = {}

    X_cols_norm = [f"{c}_norm" for c in feature_cols]
    y_norm_col = f"{target_col}_norm"

    # Chuẩn bị cột rỗng
    for c_norm in X_cols_norm:
        df_norm[c_norm] = np.nan
    df_norm[y_norm_col] = np.nan

    for country, g in df.groupby("Country"):
        scaler_x = StandardScaler()
        scaler_y = StandardScaler()

        X_raw = g[feature_cols].values.astype(np.float32)
        y_raw = g[target_col].values.reshape(-1,1).astype(np.float32)

        X_s = scaler_x.fit_transform(X_raw)
        y_s = scaler_y.fit_transform(y_raw).flatten()

        # ghi ngược vào df_norm
        df_norm.loc[g.index, X_cols_norm] = X_s
        df_norm.loc[g.index, y_norm_col] = y_s

        scalers[country] = {"x": scaler_x, "y": scaler_y}

    return df_norm, scalers


def build_global_dataset_for_horizon(df, horizon_key, target_col, test_ratio=0.15, val_ratio=0.15):
    """
    df: fred_processed_df
    horizon_key: 'q4' hoặc 'y4'
    target_col: 'target_q_4' hoặc 'target_q_16'
    Trả về:
      X_train, X_val, X_test,
      y_train, y_val, y_test,
      meta_train, meta_val, meta_test,
      feature_cols_norm,
      scalers_per_country
    """
    # feature thô = tất cả lag/rolling/time_index
    base_feature_cols = [
        c for c in df.columns
        if (
            c.startswith("GDP_lag_")
            or c.startswith("GDP_roll_mean_")
            or c.startswith("GDP_roll_std_")
            or c == "time_index"
        )
    ]

    # Chuẩn hóa per country
    df_norm, scalers = normalize_features_per_country(df, base_feature_cols, target_col)

    feature_cols_norm = [f"{c}_norm" for c in base_feature_cols]
    target_norm_col   = f"{target_col}_norm"

    use_df = df_norm.dropna(subset=feature_cols_norm + [target_norm_col]).copy()
    use_df = use_df.sort_values(["Country","Date"]).reset_index(drop=True)

    n = len(use_df)
    test_n = int(np.ceil(n * test_ratio))
    val_n  = int(np.ceil(n * val_ratio))
    train_n = n - val_n - test_n

    train_df = use_df.iloc[:train_n]
    val_df   = use_df.iloc[train_n:train_n+val_n]
    test_df  = use_df.iloc[train_n+val_n:]

    X_train = train_df[feature_cols_norm].values.astype(np.float32)
    y_train = train_df[target_norm_col].values.astype(np.float32)

    X_val   = val_df[feature_cols_norm].values.astype(np.float32)
    y_val   = val_df[target_norm_col].values.astype(np.float32)

    X_test  = test_df[feature_cols_norm].values.astype(np.float32)
    y_test  = test_df[target_norm_col].values.astype(np.float32)

    meta_train = train_df[["Country","Date"]]
    meta_val   = val_df[["Country","Date"]]
    meta_test  = test_df[["Country","Date"]]

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_train": meta_train,
        "meta_val": meta_val,
        "meta_test": meta_test,
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "scalers_per_country": scalers
    }


# Chuẩn bị dữ liệu cho 2 horizon
dataset_q4 = build_global_dataset_for_horizon(fred_processed_df, "q4", "target_q_4")
dataset_y4 = build_global_dataset_for_horizon(fred_processed_df, "y4", "target_q_16")

print("dataset_q4 shapes:", dataset_q4["X_train"].shape, dataset_q4["X_test"].shape)
print("dataset_y4 shapes:", dataset_y4["X_train"].shape, dataset_y4["X_test"].shape)
import time
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return mae, rmse

def create_sequences(X, y, timesteps=4):
    X_seq, y_seq = [], []
    for i in range(len(X) - timesteps):
        X_seq.append(X[i:i+timesteps])
        y_seq.append(y[i+timesteps])
    return np.array(X_seq), np.array(y_seq)

def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='gelu'),
        layers.LayerNormalization(),
        layers.Dropout(0.1),
        layers.Dense(128, activation='gelu'),
        layers.LayerNormalization(),
        layers.Dropout(0.1),
        layers.Dense(64, activation='gelu'),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber()
    )
    return model

def build_rnn(input_dim, timesteps=4):
    model = models.Sequential([
        layers.Input(shape=(timesteps, input_dim)),
        layers.Bidirectional(layers.LSTM(128, return_sequences=True,
                                         activation="tanh",
                                         recurrent_dropout=0.2)),
        layers.LSTM(64, return_sequences=False,
                    activation="tanh",
                    recurrent_dropout=0.1),
        layers.LayerNormalization(),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.Huber()
    )
    return model
def train_and_eval_models_for_horizon(dataset, horizon_key):
    """
    dataset: dict từ build_global_dataset_for_horizon
    horizon_key: 'q4' hoặc 'y4'
    Return:
      results_list: list metrics cho từng model
      trained_models: dict model_name -> info {model, scalers_per_country, ...}
    """

    X_train = dataset["X_train"]
    y_train = dataset["y_train"]
    X_val   = dataset["X_val"]
    y_val   = dataset["y_val"]
    X_test  = dataset["X_test"]
    y_test  = dataset["y_test"]

    feature_cols_norm = dataset["feature_cols_norm"]
    scalers_per_country = dataset["scalers_per_country"]
    target_col = dataset["target_col"]

    results_list = []
    trained_models = {}

    # 1. MLP
    mlp = build_mlp(X_train.shape[1])
    es = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    t0 = time.time()
    mlp.fit(X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=500, batch_size=64, verbose=0,
            callbacks=[es])
    train_time = time.time() - t0
    y_pred_mlp = mlp.predict(X_test, verbose=0).flatten()
    mae, rmse = compute_metrics(y_test, y_pred_mlp)
    results_list.append({
        "horizon": horizon_key,
        "model": "mlp",
        "mae": mae,
        "rmse": rmse,
        "train_time_s": train_time
    })
    trained_models["mlp"] = {
        "model": mlp,
        "type": "mlp",
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "timesteps": None,
        "scalers_per_country": scalers_per_country
    }

    # 2. RNN (LSTM)
    TIMESTEPS = 4
    Xtr_seq, ytr_seq = create_sequences(X_train, y_train, TIMESTEPS)
    Xvl_seq, yvl_seq = create_sequences(X_val,   y_val,   TIMESTEPS)
    Xte_seq, yte_seq = create_sequences(X_test,  y_test,  TIMESTEPS)

    rnn = build_rnn(input_dim=Xtr_seq.shape[2], timesteps=TIMESTEPS)
    es2 = callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
    t0 = time.time()
    rnn.fit(Xtr_seq, ytr_seq,
            validation_data=(Xvl_seq, yvl_seq),
            epochs=500, batch_size=64, verbose=0,
            callbacks=[es2])
    train_time = time.time() - t0
    y_pred_rnn = rnn.predict(Xte_seq, verbose=0).flatten()
    mae, rmse = compute_metrics(yte_seq, y_pred_rnn)
    results_list.append({
        "horizon": horizon_key,
        "model": "rnn",
        "mae": mae,
        "rmse": rmse,
        "train_time_s": train_time
    })
    trained_models["rnn"] = {
        "model": rnn,
        "type": "rnn",
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "timesteps": TIMESTEPS,
        "scalers_per_country": scalers_per_country
    }

    # 3. Decision Tree
    dt = DecisionTreeRegressor(max_depth=None, random_state=42)
    t0 = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - t0
    y_pred_dt = dt.predict(X_test)
    mae, rmse = compute_metrics(y_test, y_pred_dt)
    results_list.append({
        "horizon": horizon_key,
        "model": "dt",
        "mae": mae,
        "rmse": rmse,
        "train_time_s": train_time
    })
    trained_models["dt"] = {
        "model": dt,
        "type": "tree",
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "timesteps": None,
        "scalers_per_country": scalers_per_country
    }

    # 4. Random Forest
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    t0 = time.time()
    rf.fit(X_train, y_train)
    train_time = time.time() - t0
    y_pred_rf = rf.predict(X_test)
    mae, rmse = compute_metrics(y_test, y_pred_rf)
    results_list.append({
        "horizon": horizon_key,
        "model": "rf",
        "mae": mae,
        "rmse": rmse,
        "train_time_s": train_time
    })
    trained_models["rf"] = {
        "model": rf,
        "type": "tree",
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "timesteps": None,
        "scalers_per_country": scalers_per_country
    }

    # 5. XGBoost
    xgb = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    t0 = time.time()
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    train_time = time.time() - t0
    y_pred_xgb = xgb.predict(X_test)
    mae, rmse = compute_metrics(y_test, y_pred_xgb)
    results_list.append({
        "horizon": horizon_key,
        "model": "xgb",
        "mae": mae,
        "rmse": rmse,
        "train_time_s": train_time
    })
    trained_models["xgb"] = {
        "model": xgb,
        "type": "boost",
        "feature_cols_norm": feature_cols_norm,
        "target_col": target_col,
        "timesteps": None,
        "scalers_per_country": scalers_per_country
    }

    return results_list, trained_models
results_q4, models_q4 = train_and_eval_models_for_horizon(dataset_q4, "q4")
results_y4, models_y4 = train_and_eval_models_for_horizon(dataset_y4, "y4")

# Gộp
results_all = pd.DataFrame(results_q4 + results_y4)
display(results_all)

# Gom tất cả model vào dict chung để tiện dùng khi predict country
trained_global_models = {
    "q4": models_q4,
    "y4": models_y4
}
def predict_country_series(country_df, model_info):
    """
    country_df: df của 1 nước từ fred_processed_df
    model_info: 1 model trong trained_global_models[hkey][model_name]
    Trả về DataFrame với Date, y_true (GDP tương lai), y_pred (GDP dự báo)
    """

    model = model_info["model"]
    model_type = model_info["type"]
    feat_cols_norm = model_info["feature_cols_norm"]
    target_col = model_info["target_col"]
    timesteps = model_info["timesteps"]
    scalers_per_country = model_info["scalers_per_country"]

    country = country_df["Country"].iloc[0]
    scalers = scalers_per_country.get(country, None)
    if scalers is None:
        return None

    scaler_x = scalers["x"]
    scaler_y = scalers["y"]

    # dựng lại feature thô theo đúng thứ tự feat_cols_norm (bỏ hậu tố _norm)
    feat_cols_raw = [c.replace("_norm","") for c in feat_cols_norm]

    usable = country_df.dropna(subset=feat_cols_raw + [target_col]).copy()
    if usable.empty:
        return None

    # scale X và y_true để so sánh
    X_raw = usable[feat_cols_raw].values.astype(np.float32)
    y_true_raw = usable[target_col].values.astype(np.float32)

    X_scaled = scaler_x.transform(X_raw)

    # dự báo
    if model_type == "rnn":
        # RNN cần sequence
        X_seq, _ = create_sequences(X_scaled, y_true_raw, timesteps)
        if X_seq.shape[0] == 0:
            return None
        y_pred_scaled = model.predict(X_seq, verbose=0).flatten()
        # align lại thời gian: sau sequence N timesteps mới có dự báo
        aligned_dates = usable["Date"].values[timesteps:]
        aligned_true  = y_true_raw[timesteps:]
    else:
        # MLP / tree / RF / XGB -> input 2D
        y_pred_scaled = model.predict(X_scaled)
        aligned_dates = usable["Date"].values
        aligned_true  = y_true_raw

    # inverse transform y_pred_scaled về GDP thật scale
    y_pred_real = scaler_y.inverse_transform(y_pred_scaled.reshape(-1,1)).flatten()

    out = pd.DataFrame({
        "Date": aligned_dates,
        "y_true": aligned_true,
        "y_pred": y_pred_real
    })
    return out


def generate_all_country_predictions(fred_processed_df, trained_global_models):
    """
    Trả về:
      predictions[(country, horizon, model_name)] = df với (Date, y_true, y_pred)
    """
    predictions = {}

    for hkey, model_pack in trained_global_models.items():
        for model_name, model_info in model_pack.items():
            target_col = model_info["target_col"]  # target_q_4 hoặc target_q_16

            for country, g in fred_processed_df.groupby("Country"):
                g_sorted = g.sort_values("Date").reset_index(drop=True)
                res = predict_country_series(g_sorted, model_info)
                if res is None or res.empty:
                    continue

                # y_true hiện tại đang ở "giá trị thật tương lai" hay ở scale chuẩn hoá?
                # y_true trong res là y_true_raw (GDP target future RÀNG BUỘC),
                # tức là giá trị thực tế target_q_h (không inverse nữa)
                # => y_true đang đúng thang GDP, good.

                predictions[(country, hkey, model_name)] = res

    return predictions

predictions_by_country = generate_all_country_predictions(
    fred_processed_df,
    trained_global_models
)

print("Prediction keys sample:")
print(list(predictions_by_country.keys())[:5])
import matplotlib.pyplot as plt

def plot_country_forecasts(country, horizon_key, predictions_by_country):
    plt.figure(figsize=(10,5))
    legend_done = []

    # gom từng model
    for model_name in ["mlp","rnn","dt","rf","xgb"]:
        key = (country, horizon_key, model_name)
        if key not in predictions_by_country:
            continue
        pdf = predictions_by_country[key].copy().sort_values("Date")
        if pdf.empty:
            continue

        # vẽ y_true chỉ một lần
        if "actual" not in legend_done:
            plt.plot(pdf["Date"], pdf["y_true"], color="black", linewidth=2, label="Actual target")
            legend_done.append("actual")

        plt.plot(
            pdf["Date"],
            pdf["y_pred"],
            "--",
            label=f"{model_name.upper()} forecast",
            alpha=0.8
        )

    plt.title(f"{country} – Forecast horizon {horizon_key}")
    plt.xlabel("Date")
    plt.ylabel("Future GDP (target value)")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Ví dụ: United States, dự báo 4 quý (q4)
plot_country_forecasts("United States", "q4", predictions_by_country)

# Ví dụ: United States, dự báo 16 quý (y4)
plot_country_forecasts("United States", "y4", predictions_by_country)

# Country-level evaluation removed per request.
# Previously this block computed evaluate_country_metrics(predictions_by_country)
# and displayed results_country and avg_results. Removed to skip per-country eval.

results_country = None
avg_results = None


In [ ]:
# 8) Global evaluation summary (no country-level block as requested)
def summarize_global_results(results_q4, results_y4):
    df_q4 = pd.DataFrame(results_q4).copy()
    df_y4 = pd.DataFrame(results_y4).copy()
    summary = pd.concat([df_q4, df_y4], ignore_index=True)
    if 'test_time_s' not in summary.columns: summary['test_time_s'] = np.nan
    return summary.sort_values(['horizon','rmse'])

global_model_summary = summarize_global_results(results_q4, results_y4)
display(global_model_summary)

def pick_best_worst(summary_df):
    out_rows = []

    for hz, g in summary_df.groupby("horizon"):
        # best = RMSE nhỏ nhất, worst = RMSE lớn nhất
        best_row = g.loc[g["rmse"].idxmin()]
        worst_row = g.loc[g["rmse"].idxmax()]

        out_rows.append({
            "horizon": hz,
            "best_model": best_row["model"],
            "best_rmse": best_row["rmse"],
            "best_mae": best_row.get("mae", np.nan),
            "best_train_time_s": best_row.get("train_time_s", np.nan),
            "best_test_time_s": best_row.get("test_time_s", np.nan),

            "worst_model": worst_row["model"],
            "worst_rmse": worst_row["rmse"],
            "worst_mae": worst_row.get("mae", np.nan),
            "worst_train_time_s": worst_row.get("train_time_s", np.nan),
            "worst_test_time_s": worst_row.get("test_time_s", np.nan)
        })

    return pd.DataFrame(out_rows)

best_worst_models = pick_best_worst(global_model_summary)
display(best_worst_models)